# Mountain Car avec un DQN

Hello here 👋​. On est reparti pour un nouvel agorithme. Naturellement après avoir quitté Q-Learning on passe à son grand frère spirituel. Je vous l'avait teasé et vous savez un peu ce dont il s'agit. Sans plus attendre allons vers le vif du sujet
 
Ce notebook construit, entraîne et évalue un agent d’apprentissage par renforcement capable de résoudre l’environnement `MountainCar-v0` de Gymnasium.

L’objectif n’est pas seulement d’obtenir un agent qui réussit : nous allons suivre les étapes qui permettent de comprendre **ce que l’agent observe**, **comment il choisit une action**, **comment le DQN apprend**, puis **comment nous évaluons et enregistrons le résultat**.

> **Question directrice :** comment apprendre à une voiture à atteindre le sommet alors qu’elle ne possède pas assez de puissance pour y arriver en ligne droite ?

## Parcours du notebook

1. Découvrir l’environnement et ses espaces d’observation et d’action.
2. Formaliser le problème comme un processus de décision markovien.
3. Construire un réseau qui approxime la fonction $Q$.
4. Utiliser un replay buffer et une cible de Bellman pour entraîner le DQN.
5. Évaluer l’agent, visualiser une partie et enregistrer une vidéo.

L’implémentation est réalisée avec **Gymnasium**, **PyTorch** et **NumPy**.

## 1. Préparer l’environnement de travail

Nous importons les bibliothèques nécessaires :

- **Gymnasium** fournit l’environnement et son API `reset` / `step`.
- **PyTorch** permet de définir et d’entraîner le réseau de neurones.
- **NumPy** facilite la manipulation des observations.
- **Stable-Baselines3** est importé pour pouvoir comparer cette implémentation avec des algorithmes prêts à l’emploi ; dans la suite, l’agent principal est toutefois codé avec PyTorch.

In [1]:
import random
from collections import deque
from tqdm.notebook import tqdm as tqdm_n
import gymnasium as gym

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributions as distributions
import numpy as np

from stable_baselines3 import PPO, DQN, A2C, SAC


## 2. Découvrir l’environnement

In [2]:
env = gym.make("MountainCar-v0")

In [3]:
print(f"Espace d'observation: {env.observation_space}")

print(f"Espace d'action: {env.action_space}")

Espace d'observation: Box([-1.2  -0.07], [0.6  0.07], (2,), float32)
Espace d'action: Discrete(3)


### 2.1. Observations, actions et dynamique

L’environnement `MountainCar-v0` modélise une voiture placée dans une vallée. À chaque étape, l’agent reçoit une observation, choisit une action et obtient une récompense ainsi qu’un nouvel état.

#### Observation

L’observation est un vecteur de deux valeurs continues :

- la **position** $x$, comprise entre $-1,2$ et $0,6$ ;
- la **vitesse** $v$, comprise entre $-0,07$ et $0,07$.

L’état fourni au réseau est donc :

$$s_t = (x_t, v_t)$$

#### Actions

L’espace d’action est discret et contient trois actions :

- `0` : accélérer vers la gauche ;
- `1` : ne pas accélérer ;
- `2` : accélérer vers la droite.

#### Dynamique

La voiture ne peut pas atteindre directement le sommet droit. Elle doit d’abord prendre de l’élan en oscillant entre les deux pentes. La dynamique simplifiée est :

$$v_{t+1} = v_t + (a_t - 1) \times \text{force} - \cos(3x_t) \times \text{gravity}$$

$$x_{t+1} = x_t + v_{t+1}$$

avec `force = 0.001` et `gravity = 0.0025`. L’épisode s’arrête lorsque la voiture atteint le sommet ou après 200 étapes.

La récompense vaut généralement `-1` à chaque étape : l’agent doit donc atteindre l’objectif le plus rapidement possible.

### 2.2. Pourquoi choisir un DQN ?

Le problème possède un espace d’action **discret** : l’agent doit choisir l’une des trois actions disponibles. Un Deep Q-Network est donc un choix naturel, car il approxime directement la valeur de chaque action possible à partir de l’état courant.

Le DQN est particulièrement adapté ici pour deux raisons :

- les récompenses sont peu informatives à court terme, puisque l’agent reçoit principalement `-1` ;
- le **replay buffer** permet de réutiliser les expériences et de réduire la dépendance entre deux transitions successives.

L’agent apprendra une approximation de $Q(s,a)$, c’est-à-dire la qualité attendue de l’action $a$ lorsqu’il se trouve dans l’état $s$.

In [4]:
class PolicyNetwork(nn.Module):
    def __init__(self, input_dim=2, hidden_dim=64, output_dim=3):
        super().__init__()
        self.layer1 = nn.Linear(input_dim, hidden_dim)
        self.layer2 = nn.Linear(hidden_dim, output_dim)
    def forward(self, x):
        x = self.layer1(x)
        x = F.relu(x)
        x = self.layer2(x)
        return x

### 3.1. Le replay buffer

Un agent apprend à partir de transitions de la forme :

$$ (s_t, a_t, r_{t+1}, s_{t+1}, \text{done}) $$

Le **replay buffer** mémorise ces transitions au fil des épisodes. Pendant l’entraînement, un mini-lot est tiré aléatoirement dans cette mémoire.

Cette stratégie permet de :

- réutiliser plusieurs fois les expériences passées ;
- mélanger des transitions provenant de moments différents ;
- éviter que le réseau apprenne uniquement à partir d’états successifs très corrélés.

La classe suivante stocke les transitions et les convertit en tenseurs PyTorch au moment de l’apprentissage.

In [5]:
class ReplayBuffer:
    def __init__(self, capacity=50000):
        self.buffer = deque(maxlen=capacity)
    
    def __len__(self):
        return len(self.buffer)
    def push(self, state, action, reward, next_state, done):
        # On ajoute l'expérience à chaque pas de temps
        self.buffer.append((state, action, reward, next_state, done))
    
    def sample(self, batch_size=64):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        
        return (
            torch.FloatTensor(states),
            torch.LongTensor(actions),
            torch.FloatTensor(rewards),
            torch.FloatTensor(next_states),
            torch.FloatTensor(dones)
        )
        
    

### 3.2. Exploration et exploitation

À chaque étape, l’agent doit arbitrer entre deux comportements :

- **explorer** : essayer une action aléatoire pour découvrir de nouvelles trajectoires ;
- **exploiter** : choisir l’action dont la valeur $Q$ estimée est la plus élevée.

Nous utilisons une stratégie **$\varepsilon$-greedy** : avec une probabilité $\varepsilon$, l’agent explore ; sinon, il exploite son réseau.

Au début, $\varepsilon$ est élevé afin de favoriser la découverte. Il diminue progressivement jusqu’à une valeur minimale, ce qui laisse toujours une petite part d’exploration.

### 3.3. Mise à jour des paramètres

Le DQN utilise deux réseaux :

- le **réseau principal** $Q_{\theta}$, mis à jour à chaque mini-lot ;
- le **réseau cible** $Q_{\theta^-}$, conservé temporairement plus stable et synchronisé périodiquement avec le réseau principal.

Pour une transition donnée, le réseau principal estime la valeur de l’action réellement exécutée :

$$Q_{\theta}(s_t, a_t)$$

La cible de Bellman est calculée à partir de la récompense et de la meilleure action estimée dans l’état suivant :

$$Y_t = r_{t+1} + \gamma (1 - \text{done}) \max_{a'} Q_{\theta^-}(s_{t+1}, a')$$

Le terme $(1 - \text{done})$ annule la valeur future lorsque l’épisode est terminé. Le réseau est entraîné pour réduire l’écart entre sa prédiction et cette cible :

$$\mathcal{L}(\theta) = \left(Q_{\theta}(s_t,a_t) - Y_t\right)^2$$

La fonction `update_dqn` applique cette mise à jour sur un mini-lot tiré du replay buffer.

In [6]:
def update_dqn(policy_net:PolicyNetwork, target_net:PolicyNetwork, optimizer, replay_buffer:ReplayBuffer, batch_size, gamma):

    if len(replay_buffer) < batch_size:
        return

    # On récupère un batch
    states, actions, rewards, next_states, dones = replay_buffer.sample(batch_size)

    # Q(s,a) : ce que pense actuellement le Policy Network
    q_values = policy_net(states)
    q_values = q_values.gather(1, actions.unsqueeze(1)).squeeze(1)

    # Cible : r + γ max Q(s',a')
    with torch.no_grad():
        next_q_values = target_net(next_states).max(1).values
        targets = rewards + gamma * next_q_values * (1 - dones)

    # Erreur
    loss = F.mse_loss(q_values, targets)

    # Apprentissage
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    return loss.item()

## 4. Entraîner l’agent

La boucle d’entraînement répète les étapes fondamentales d’un algorithme de renforcement :

1. réinitialiser l’environnement au début de chaque épisode ;
2. choisir une action avec la stratégie **$\varepsilon$-greedy** ;
3. exécuter l’action avec `env.step(action)` ;
4. stocker la transition dans le replay buffer ;
5. mettre à jour le réseau principal lorsque le buffer contient assez d’expériences ;
6. synchroniser périodiquement le réseau cible ;
7. réduire progressivement $\varepsilon$.

Nous entraînons l’agent pendant 600 épisodes, avec au maximum 200 étapes par épisode. Le replay buffer peut contenir 50 000 transitions et les mini-lots contiennent 64 transitions.

Le suivi de la récompense et de la valeur de $\varepsilon$ permet d’observer à la fois la progression de l’agent et la diminution de son exploration.

In [17]:

def train():
    
    policy_net = PolicyNetwork()
    env = gym.make("MountainCar-v0")

    # Hyperparamètres
    gamma = 0.99
    lr = 1e-3
    batch_size = 64
    episodes = 1500

    epsilon = 1.0
    epsilon_min = 0.05
    epsilon_decay = 0.998

    target_update = 100


    # Réseaux


 
    target_net = PolicyNetwork()

    target_net.load_state_dict(policy_net.state_dict())

    optimizer = torch.optim.Adam(
        policy_net.parameters(),
        lr=lr
    )

    replay_buffer = ReplayBuffer(50_000)

    step = 0

    # BOUCLE D'ENTRAÎNEMENT

    for episode in tqdm_n(range(episodes)):

        state, _ = env.reset()
        done = False
        total_reward = 0

        while not done:

            # 1. Choisir une action
            if random.random() < epsilon:

                # Exploration
                action = env.action_space.sample()

            else:

                # Exploitation
                state_tensor = torch.tensor(
                    state,
                    dtype=torch.float32
                ).unsqueeze(0)

                with torch.no_grad():
                    action = policy_net(state_tensor).argmax().item()

            # 2. Jouer l'action
            next_state, reward, terminated, truncated, _ = env.step(action)

            done = terminated or truncated

            # 3. Stocker l'expérience
            replay_buffer.push(
                state,
                action,
                reward,
                next_state,
                done
            )

            # 4. Passer à l'état suivant
            state = next_state
            total_reward += reward

            step += 1

            # 5. Entraîner le Policy Network
            if len(replay_buffer) >= batch_size:

                update_dqn(
                    policy_net,
                    target_net,
                    optimizer,
                    replay_buffer,
                    batch_size,
                    gamma
                )

            # 6. Mettre à jour le Target Network
            if step % target_update == 0:

                target_net.load_state_dict(
                    policy_net.state_dict()
                )

        # 7. Diminuer progressivement l'exploration
        epsilon = max(
            epsilon_min,
            epsilon * epsilon_decay
        )

        # 8. Affichage
        print(
            f"Episode {episode + 1} | "
            f"Reward: {total_reward:.0f} | "
            f"Epsilon: {epsilon:.3f}"
        )

    env.close()
    
    return policy_net

In [18]:
print(f"Entrainement de notre politique")
policy_net = train()

Entrainement de notre politique


  0%|          | 0/1500 [00:00<?, ?it/s]

Episode 1 | Reward: -200 | Epsilon: 0.998
Episode 2 | Reward: -200 | Epsilon: 0.996
Episode 3 | Reward: -200 | Epsilon: 0.994
Episode 4 | Reward: -200 | Epsilon: 0.992
Episode 5 | Reward: -200 | Epsilon: 0.990
Episode 6 | Reward: -200 | Epsilon: 0.988
Episode 7 | Reward: -200 | Epsilon: 0.986
Episode 8 | Reward: -200 | Epsilon: 0.984
Episode 9 | Reward: -200 | Epsilon: 0.982
Episode 10 | Reward: -200 | Epsilon: 0.980
Episode 11 | Reward: -200 | Epsilon: 0.978
Episode 12 | Reward: -200 | Epsilon: 0.976
Episode 13 | Reward: -200 | Epsilon: 0.974
Episode 14 | Reward: -200 | Epsilon: 0.972
Episode 15 | Reward: -200 | Epsilon: 0.970
Episode 16 | Reward: -200 | Epsilon: 0.968
Episode 17 | Reward: -200 | Epsilon: 0.967
Episode 18 | Reward: -200 | Epsilon: 0.965
Episode 19 | Reward: -200 | Epsilon: 0.963
Episode 20 | Reward: -200 | Epsilon: 0.961
Episode 21 | Reward: -200 | Epsilon: 0.959
Episode 22 | Reward: -200 | Epsilon: 0.957
Episode 23 | Reward: -200 | Epsilon: 0.955
Episode 24 | Reward:

## 5. Analyser les résultats de l’entraînement

La récompense de `MountainCar-v0` est généralement égale à `-1` par étape. Comme un épisode est limité à 200 étapes, une récompense de `-200` signifie que la voiture n’a pas atteint l’objectif avant la limite de temps. Plus la récompense est proche de zéro, plus l’agent a terminé rapidement.

### 5.1. Apparition d’un premier signal d’apprentissage

Pendant plus de 700 épisodes même avec un epssilon a 0.25, l’agent explore encore largement l’environnement et n'a aucun vrai signal à exploiter.

On note quand même de rares réussites. L'agent atteint la cible de rare fois ce qui permet de créer des cas ou le réseau à quelque chose à apprendre



Il s’agit toutefois de succès ponctuels : avec $\varepsilon \approx 0,22$, l’agent continue à explorer et ses performances peuvent encore varier fortement.

### 5.2. Le vrai déclic

A partir des épisode 775 dans mon cas, on osberve une amélioration nette; plus de -200 à tous vas mais de plus en plus de cas ou l'agent atteint l'objectif et celà va se poursuivre jusqu'a la fin de l'entraînement où il réussit presque toujours à atteindre la cible avec une récompense pouvant varier de -94 à -196


### 5.3. Évaluation sur 100 épisodes sans exploration

Pour mesurer la qualité réelle de la politique, nous l’évaluons sur 100 épisodes indépendants avec une stratégie gloutonne : $\varepsilon = 0$. Les indicateurs calculés sont :

| Indicateur | Résultat de la dernière évaluation |
| --- | ---: |
| Récompense moyenne | `-146.56` |
| Récompense médiane | `-153` |
| Variance | `577,99` |
| Écart-type | `24.08` |
| Meilleure récompense | `-91` |
| Pire récompense | `-196` |
| Nombre moyen d’étapes | `147` |
| Taux de réussite | `100 %` |



La politique a appris un signal utile et efficace et est capable de résoudre plus ou moins aisément le problème de Moutain Car. Il a appris qu'il fallait reculer et prendre de l'élan avant de foncer pour atteindre la cible



La médiane de `-153`, plus faible que la moyenne de `-146`, indique que de nombreux épisodes sont encore longs, tandis que quelques épisodes très réussis améliorent la moyenne. L’écart-type de `24.08` confirme une variabilité importante. (PS: ce entrainement est beacoup mieux parce que sur une précédente tentative j'avais obtenu une variance de plus de 2400 et un taux de réussite d'environ 54%)


**Conclusion :** le DQN a bien appris une stratégie utile, mais certains épisodes sont encore assez long. Faire du rewards shapping pourrait nous aider à obtenir un signal plus exploitable pour réduire notre écart type. Cependant, bien que le modèle semble être suffisant performant, je reste dubitatif sur sa stabilité. On obtient une pire récompense de -196 ce qui est à 4 steps de l'arret d'un épisode. Il se peut qu'il y ait des épisodes qui nécéssiteraient un peu plus pour atteindre la cible.

Par contre je suis assez fier du résultats. Ca reste une implémentation from scratch très élégante. Augmenter le nombre d'itérations pourrait peut être amené à de meilleurs résultats (But anyway :)

In [19]:
# Évaluation statistique de la politique apprise

n_episodes = 100
eval_env = gym.make("MountainCar-v0")
rewards = []
steps_per_episode = []
successes = 0

policy_net.eval()
try:
    for episode in range(n_episodes):
        state, _ = eval_env.reset()
        total_reward = 0.0
        steps = 0
        terminated = False
        truncated = False

        while not (terminated or truncated):
            state_tensor = torch.tensor(
                state,
                dtype=torch.float32
            ).unsqueeze(0)

            with torch.no_grad():
                action = policy_net(state_tensor).argmax(dim=1).item()

            state, reward, terminated, truncated, _ = eval_env.step(action)
            total_reward += reward
            steps += 1

        rewards.append(total_reward)
        steps_per_episode.append(steps)
        successes += int(terminated)
finally:
    eval_env.close()

rewards_array = np.array(rewards, dtype=np.float32)
steps_array = np.array(steps_per_episode, dtype=np.int32)

print(f"Nombre d'épisodes évalués : {n_episodes}")
print(f"Récompense moyenne       : {rewards_array.mean():.2f}")
print(f"Récompense médiane       : {np.median(rewards_array):.2f}")
print(f"Variance des récompenses : {rewards_array.var():.2f}")
print(f"Écart-type               : {rewards_array.std():.2f}")
print(f"Meilleure récompense     : {rewards_array.max():.0f}")
print(f"Pire récompense          : {rewards_array.min():.0f}")
print(f"Étapes moyennes          : {steps_array.mean():.2f}")
print(f"Réussites                : {successes}/{n_episodes}")
print(f"Taux de réussite         : {successes / n_episodes:.1%}")

Nombre d'épisodes évalués : 100
Récompense moyenne       : -146.56
Récompense médiane       : -153.00
Variance des récompenses : 577.99
Écart-type               : 24.04
Meilleure récompense     : -91
Pire récompense          : -196
Étapes moyennes          : 146.56
Réussites                : 100/100
Taux de réussite         : 100.0%


In [21]:
# Visualisation

env_visuel = gym.make(
    "MountainCar-v0",
    render_mode="human"
)

state, _ = env_visuel.reset()

done = False
try:
    while not done:

        state_tensor = torch.tensor(
            state,
            dtype=torch.float32
        ).unsqueeze(0)

        with torch.no_grad():
            action = policy_net(state_tensor).argmax().item()

        state, reward, terminated, truncated, _ = env_visuel.step(action)

        done = terminated or truncated

finally:
    env_visuel.close()

## 6. Visualiser une exécution

Le mode `human` pour observer. Surement la phase la plus satisfaisante. On constate bien la statégie,le comportement appris:  la voiture doit alterner ses accélérations pour accumuler suffisamment d’élan, puis atteindre le drapeau au sommet.



In [ ]:
# On enregistre le tout
torch.save(policy_net, "models/mountain-car.pt")

## 8. Enregistrer une vidéo

On conserve une trace visuelle de l’évaluation, avec `RecordVideo` tout en utilisant un environnement rendu sous forme d’images (`rgb_array`).


In [23]:
from gymnasium.wrappers import RecordVideo

env = gym.make("MountainCar-v0", render_mode="rgb_array")

env = RecordVideo(
    env, 
    video_folder="./videos", 
    episode_trigger=lambda ep_id: True,
    name_prefix="mountain-car"
)

state, _ = env.reset()
done = False

try:
    while not done:
        state_tensor = torch.tensor(state, dtype=torch.float32).unsqueeze(0)

        with torch.no_grad():
            action = policy_net(state_tensor).argmax().item()

        state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

finally:

    env.close()

print("Vidéo enregistrée dans le dossier ./videos")

e:\cours ifri\Programmation et BD\Python\Reinforcement Learning\HF Course\.venv\Lib\site-packages\gymnasium\wrappers\rendering.py:292: UserWarning: WARN: Overwriting existing videos at e:\cours ifri\Programmation et BD\Python\Reinforcement Learning\HF Course\videos folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


Vidéo enregistrée dans le dossier ./videos


## 9. Bilan et pistes d’amélioration

Félicitation !!!!

Nous avons construit un DQN complet pour un environnement à actions discrètes : réseau de neurones, replay buffer, stratégie d’exploration, réseau cible et boucle d’entraînement.

Le résultat montre surtout l’importance de l’exploration dans Mountain Car. Pour aller plus loin, nous pourrions :

- évaluer la politique sur plusieurs autres épisodes et calculer une moyenne ;
- tracer l’évolution des récompenses ;
- utiliser une perte Huber et un réseau cible mis à jour progressivement ;
- comparer cette implémentation avec `DQN` de Stable-Baselines3 (Soon);
